# Module 10 — Test the business rule

**Before running code:** decide what the answer should be.

Business rule → expected answer → test → failure → repair → rerun.

**Session 1:** a guided shipping example, then Break My Code in Pair Up.  
**Session 2:** customer conversion, then the wildfire dashboard and your project.  
**If time permits:** discounts and retention.

Codex can help with Python. You decide whether the expected answer follows the rule. The supplied bugs are prepared teaching examples, not recorded AI outputs.

## Start here

GitHub is for reading. To run this notebook, open your cloned **10-test** folder in VS Code.

1. In the terminal, run `uv sync --extra notebook`.
2. Open this notebook and select **Select Kernel → Python Environments → .venv**.
3. Run the two setup cells below.

**Where do I write?** This notebook gives the instructions and runs pytest. The setup cells create `.py` files under `pytest_business_lab/`. Open those files in VS Code’s Explorer to edit them. Save with **Ctrl+S** (Windows) or **Cmd+S** (Mac), then run the notebook’s test cell again.

**Two different files:** the implementation contains the function; `test_*.py` contains its tests. Changing one does not change the other.

Starter cells create files only once. Rerunning them preserves your edits; editing the quoted starter text does **not** change files already created. To start over, rename that activity’s folder, then rerun its setup cell.

Response cells are marked collapsed; some viewers may still display them. Worked solutions are in the instructor notebook.

In [ ]:
import sys
import subprocess
from pathlib import Path
import pytest
import pandas as pd

REPO = Path.cwd()
if not (REPO / "wildfire_dashboard").is_dir():
    raise RuntimeError("Open the notebook from the cloned 10-test folder.")
ROOT = REPO / "pytest_business_lab"
ROOT.mkdir(exist_ok=True)
print("Python:", sys.executable)
print("pytest:", pytest.__version__)
print("Activity files:", ROOT)

In [ ]:
import shutil

def seed(path, content):
    if path.exists():
        print("Keeping your existing file:", path.relative_to(REPO))
    else:
        path.write_text(content, encoding="utf-8")

def run_tests(target):
    # A fresh Python process reads your saved files on every run.
    for cache in ROOT.rglob("__pycache__"):
        shutil.rmtree(cache)
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--tb=short", str(REPO / target)],
        cwd=REPO, capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)
    print("0 = passed; 1 = a test failed; 2–4 = interrupted or unable to run; 5 = no tests found.")
    return result.returncode

## 1. Shipping — a guided test

**Rule:** shipping costs $5 below a $50 subtotal and $0 at $50 or above. The input is the merchandise subtotal **after discounts**, excluding tax. It is already validated as nonnegative.

**Predict first.** What should shipping cost at $20, $49.99, $50, and $100? Write your answers before running the next cells.

The function has a prepared mistake. The initial tests check only $20 and $100. Will those tests notice it?

In [ ]:
activity = ROOT / '01_shipping'
activity.mkdir(exist_ok=True)
seed(activity / 'shipping.py', """def shipping_fee(post_discount_subtotal):
    return 0 if post_discount_subtotal > 50 else 5
""")
seed(activity / 'test_shipping.py', """from shipping import shipping_fee

def test_common_orders():
    assert shipping_fee(20) == 5
    assert shipping_fee(100) == 0
""")
print("Edit these files:", activity)

In [ ]:
run_tests("pytest_business_lab/01_shipping")

### Add the missing case

Open `pytest_business_lab/01_shipping/test_shipping.py`. Add this test below the existing one:

```python
def test_at_free_shipping_cutoff():
    assert shipping_fee(50) == 0
```

`test_` tells pytest to collect the function. `assert` checks a claim. Here the left side calls the code; the right side is the expected answer from the shipping rule.

The collapsed cell below makes the same edit for this guided example. **Use either the file edit or the cell.** The cell replaces this test file with the two tests shown here.

In [ ]:
(ROOT / '01_shipping' / 'test_shipping.py').write_text("""from shipping import shipping_fee

def test_common_orders():
    assert shipping_fee(20) == 5
    assert shipping_fee(100) == 0

def test_at_free_shipping_cutoff():
    assert shipping_fee(50) == 0
""", encoding="utf-8")

In [ ]:
run_tests("pytest_business_lab/01_shipping")

### Read → repair → rerun

Find the failing test name and the line containing `assert`. Which amount did the function return? Which amount did the rule require?

Open `pytest_business_lab/01_shipping/shipping.py`. Fix the cutoff, save, then rerun the next cell. **Keep the expected answer unchanged.** If it still fails, check that you saved the `.py` file rather than editing its starter text above.

After the repair, add your $49.99 case. Explain why both sides of the cutoff matter.

In [ ]:
# Your explanation: actual result, expected result, and the rule that decides it.

In [ ]:
run_tests("pytest_business_lab/01_shipping")

## 2. Break My Code — use Pair Up

Open [Pair Up](https://dso576.vercel.app/pair), sign in with your usual course code, select **Break My Code**, and join the instructor’s activity. Wait for the round to start.

**Hotel rule:** flexible bookings receive a full refund at least 48 hours before check-in, half at 24 to less than 48 hours, and zero below 24 hours. Nonrefundable bookings receive zero. Amounts and hours are nonnegative; time zones are handled before this function is called.

Use the function name and inputs shown in the activity. The hotel preset uses `cancellation_refund(prepaid_amount, hours_before_checkin, rate_type)`.

- **Developer:** implement the stated rule and submit your code.
- **Tester:** choose inputs and expected refunds from the rule, then write pytest tests. Submit before looking at the developer’s code.
- Once both submit, compare the files. Save the downloaded `module_under_test.py` and `test_submission.py` in one folder. Open a terminal in that folder and run the command shown in Pair Up. Share the actual output and exit code.
- Discuss one failure, repair the implementation, and rerun. When the instructor starts round 2, switch roles.

**Individual mode:** everyone tests the instructor’s code instead. There is no partner or role switch.

Codex is allowed in either role. Choose expected answers before asking it for test syntax. Python runs on your computer; Pair Up shares submissions and reported results.

**Debrief:** Which test would fail for a plausible but wrong implementation? What would a passing result still leave unchecked?

In [ ]:
# Record one input, your expected refund, and the business rule behind it.

## 3. Customer conversion — count people, not transactions

**Rule:** among distinct visitors in an experiment, what fraction purchased at least once? Each visitor counts once. For this exercise, the visitor list is nonempty and every order belongs to a listed visitor.

**Predict first:** visitors are A, B, C, D. Orders are from A, A, C. What is the numerator? The denominator? The conversion rate?

The initial test has no repeat purchases. Predict whether it will expose a function that counts orders instead of purchasers.

In [ ]:
activity = ROOT / '04_dashboard'
activity.mkdir(exist_ok=True)
seed(activity / 'metrics.py', """def conversion_rate(visitors, orders):
    return len(orders) / visitors["visitor_id"].nunique()
""")
seed(activity / 'test_metrics.py', """import pandas as pd
import pytest
from metrics import conversion_rate

def test_one_purchase():
    visitors = pd.DataFrame({"visitor_id": ["A", "B"]})
    orders = pd.DataFrame({"visitor_id": ["A"]})
    assert conversion_rate(visitors, orders) == pytest.approx(0.5)
""")
print("Edit these files:", activity)

In [ ]:
run_tests("pytest_business_lab/04_dashboard")

Open `pytest_business_lab/04_dashboard/test_metrics.py`. Add a test using A, B, C, D and orders A, A, C. Create the two DataFrames inside the test, as in the starter. Write your expected rate yourself.

Run `run_tests("pytest_business_lab/04_dashboard")`. Read the failure, fix `metrics.py`, save, and rerun. Then add another order for A: should the answer change?

`pytest.approx(expected)` allows tiny floating-point differences. It does not excuse a wrong numerator or denominator.

**What this does not decide:** the policy for no visitors or orders from outside the experiment. Agree on those rules before testing them.

In [ ]:
# Add your test in test_metrics.py, save, then uncomment:
# run_tests("pytest_business_lab/04_dashboard")

## 4. Wildfire — test a result behind the dashboard

This is the actual dashboard backend, separate from the small exercises above.

**Look first.** From the repository terminal, run:

```text
uv run --extra dashboard python -m wildfire_dashboard.html_dashboard
```

Open the printed local URL. Select California and a year within 2000–2024. Locate the fire count and drought chart. Keep the server terminal running; use this notebook to run tests.

**Predict before opening the tests:**

1. Three fires have acre values `2`, missing, and `4`. How many fires should count? What is the known acreage total?
2. Two California drought maps in January show D1-or-worse coverage of 20% and 40%. The monthly rule is the mean of the weekly maps. What should January show?
3. February has no drought observation. Should the join show missing or 0%?

Now read `tests/test_dashboard.py` and `tests/test_drought.py`. Find the assertions that check your answers. Drought categories are cumulative; do not add D1 and D2 together.

In [ ]:
run_tests("tests/test_dashboard.py")
run_tests("tests/test_drought.py")

### Add one check

In `tests/test_my_checks.py`, add a test for this new case: two fire records have missing acreage. The fire count should still be two. Use `metrics` from `wildfire_dashboard.wildfire_data`; copy the small DataFrame structure from `test_dashboard.py` and change its inputs.

Run the new test from the terminal:

```text
uv run python -m pytest tests/test_my_checks.py -q
```

A new test can pass immediately. Explain the behavior it protects. These tests check Python calculations; they do not prove that browser filters or charts display those calculations correctly.

In [ ]:
# Your check: inputs, expected result, and the dashboard behavior it protects.

## Extensions — if time permits

### A. Discounts and approximate comparisons

**Rule:** apply one discount to the eligible subtotal. Inputs are a nonnegative subtotal and a rate between 0 and 1.

Calculate the price of a $120 basket after a 15% discount. Then inspect the starter below: which test would distinguish one discount from two?

For floats, mathematically equal calculations can differ slightly: `0.1 + 0.2 == 0.3` is false. `pytest.approx(0.3)` accepts that tiny difference. It will not make an incorrect double discount pass. This exercise does not specify production currency rounding.

In [ ]:
activity = ROOT / '03_promo'
activity.mkdir(exist_ok=True)
seed(activity / 'promotion.py', """def net_price(eligible_subtotal, discount_rate):
    discounted = eligible_subtotal * (1 - discount_rate)
    return discounted * (1 - discount_rate)
""")
seed(activity / 'test_promotion.py', """import pytest
from promotion import net_price

def test_no_discount():
    assert net_price(100, 0) == pytest.approx(100)
""")
print("Edit these files:", activity)
run_tests("pytest_business_lab/03_promo")

Add your $120/15% case in `pytest_business_lab/03_promo/test_promotion.py` using `pytest.approx`. Run it, explain the failure, repair `promotion.py`, and rerun.

In [ ]:
# Write your expected price before asking Codex for the test syntax.

### B. Retention — check Codex’s proposed tests

**Rule:** retention is distinct January purchasers who also purchased in February, divided by distinct January purchasers. The January group is nonempty. Repeated transactions do not change the rate.

January: A, A, B, C. February: A, A, D. Write the expected fraction on paper.

Use the supplied implementation below. Ask Codex to propose tests from the rule **without reading or changing the implementation**. Compare its expected answers with yours before accepting them. If it misses duplicates, add that case yourself.

In [ ]:
activity = ROOT / '05_retention'
activity.mkdir(exist_ok=True)
seed(activity / 'retention.py', """def retained_fraction(january_customers, february_customers):
    return len(set(january_customers) & set(february_customers)) / len(january_customers)
""")
seed(activity / 'test_retention.py', """import pytest
from retention import retained_fraction

def test_no_one_returns():
    assert retained_fraction(["A", "B"], ["C"]) == pytest.approx(0)
""")
print("Edit these files:", activity)
run_tests("pytest_business_lab/05_retention")

Add the approved tests to `pytest_business_lab/05_retention/test_retention.py`. Run them, repair `retention.py`, and rerun. Does a passing test for “no one returns” check the denominator?

In [ ]:
# Record one proposed test you accepted, changed, or rejected, and why.

## Your full project

On **your own branch** of the team project, write pytest tests for its important behavior across the project—not only your Module 9 audit.

For each behavior, decide the expected result from a business rule or small hand-checkable dataset. Include ordinary cases and relevant boundaries, missing data, or duplicates. Do not invent requirements just to add tests.

Commit the tests and `test-report.md`. State what you checked, what failed or passed, and what remains untested. Submit the report to Brightspace.

**Exit ticket:** name one passing test that could miss a business error, and one expected answer you would check yourself before accepting Codex’s work.